# GenAI Assignment 1 23i0618 Iman
Runs the full pipeline on a free GPU: data prep -> Tasks 1-4 (Optuna + MLflow) -> evaluation -> ONNX export.
**Runtime > Change runtime type > GPU (T4)** before you start. Budgets below are for a T4; each task writes checkpoints to Google Drive so a disconnect does not lose work.

##Mount Drive and get the code

In [2]:
from google.colab import drive
drive.mount('/content/drive')
import os
WORK = '/content/drive/MyDrive/genai_a1'          # checkpoints, studies, mlflow.db and outputs live here
os.makedirs(WORK, exist_ok=True)

Mounted at /content/drive


In [4]:
#clone your GitHub repo
REPO_URL = 'https://github.com/ImaanSuleiman/genai-assignment1.git'
!git clone $REPO_URL /content/repo || (cd /content/repo && git pull)


Cloning into '/content/repo'...
fatal: could not read Username for 'https://github.com': No such device or address
/bin/bash: line 1: cd: /content/repo: No such file or directory


In [6]:
import zipfile, shutil, glob, os

dst = 'data/FS2K'   # lives Drive through the data/ symlink
if not os.path.exists(f'{dst}/anno_train.json'):
    cands = ['/content/drive/MyDrive/FS2K.zip', f'{WORK}/FS2K.zip', f'{WORK}/data/FS2K.zip']
    zp = next((c for c in cands if os.path.exists(c)), None)
    assert zp, 'FS2K.zip not found in MyDrive. If it only shows under "Shared with me", right-click it in Drive > Make a copy, then re-run.'
    print('unzipping', zp, '(takes a few minutes)')
    shutil.rmtree('/content/fs2k_tmp', ignore_errors=True)
    zipfile.ZipFile(zp).extractall('/content/fs2k_tmp')          # unzip on fast local disk first
    annos = glob.glob('/content/fs2k_tmp/**/anno_train.json', recursive=True)
    assert annos, 'anno_train.json not found inside the zip'
    shutil.copytree(os.path.dirname(annos[0]), dst, dirs_exist_ok=True)   # then copy to Drive

for sub in ['photo', 'sketch']:
    print(sub, 'files:', len(glob.glob(f'{dst}/{sub}/**/*.*', recursive=True)))
print(os.listdir(dst))

!python -m genai.prepare --fs2k

unzipping /content/drive/MyDrive/FS2K.zip (takes a few minutes)
photo files: 2104
sketch files: 2104
['sketch', 'README.pdf', 'anno_test.json', 'photo', 'anno_train.json']
/usr/bin/python3: Error while finding module specification for 'genai.prepare' (ModuleNotFoundError: No module named 'genai')


## 1. Data
**Pets** downloads automatically.

In [14]:
import os
%cd /content
REPO_URL = 'https://github.com/ImaanSuleiman/genai-assignment1.git'
!rm -rf /content/repo
!git clone $REPO_URL /content/repo
%cd /content/repo
!pip install -q -r requirements.txt
print(os.listdir('/content/repo'))

/content
Cloning into '/content/repo'...
remote: Enumerating objects: 106, done.
remote: Counting objects: 100% (106/106), done.
remote: Compressing objects: 100% (83/83), done.
remote: Total 106 (delta 14), reused 106 (delta 14), pack-reused 0 (from 0)
Receiving objects: 100% (106/106), 523.17 KiB | 30.77 MiB/s, done.
Resolving deltas: 100% (14/14), done.
/content/repo
['.dockerignore', 'src', '.git', 'README.md', 'models', 'scripts', 'app', '.gitignore', 'docs', 'docker-compose.yml', '.gitattributes', 'notebooks', 'tests', 'report', 'requirements.txt']


In [15]:
!python -m genai.prepare --pets

100% 792M/792M [23:34<00:00, 560kB/s]
100% 19.2M/19.2M [00:55<00:00, 348kB/s]
Pets: train=2944 val=736 test=3669 val_manifest=736 test_manifest=36690


In [17]:
import zipfile, shutil, glob, os

dst = 'data/FS2K'   # lives on Drive through the data/ symlink
if not os.path.exists(f'{dst}/anno_train.json'):
    cands = ['/content/drive/MyDrive/FS2K.zip', f'{WORK}/FS2K.zip', f'{WORK}/data/FS2K.zip']
    zp = next((c for c in cands if os.path.exists(c)), None)
    assert zp, 'FS2K.zip not found in MyDrive. If it only shows under "Shared with me", right-click it in Drive > Make a copy, then re-run.'
    print('unzipping', zp, '(takes a few minutes)')
    shutil.rmtree('/content/fs2k_tmp', ignore_errors=True)
    zipfile.ZipFile(zp).extractall('/content/fs2k_tmp')          # unzip on fast local disk first
    annos = glob.glob('/content/fs2k_tmp/**/anno_train.json', recursive=True)
    assert annos, 'anno_train.json not found inside the zip'
    shutil.copytree(os.path.dirname(annos[0]), dst, dirs_exist_ok=True)   # then copy to Drive

for sub in ['photo', 'sketch']:
    print(sub, 'files:', len(glob.glob(f'{dst}/{sub}/**/*.*', recursive=True)))
print(os.listdir(dst))

!python -m genai.prepare --fs2k

unzipping /content/drive/MyDrive/FS2K.zip (takes a few minutes)
photo files: 2104
sketch files: 2104
['sketch', 'README.pdf', 'anno_test.json', 'photo', 'anno_train.json']
FS2K: {'train': 899, 'val': 159, 'test': 1046, 'style_counts_train': [303, 298, 298], 'style_counts_val': [54, 52, 53], 'style_counts_test': [619, 381, 46]}


## 2. Task 1: universal autoencoder (Optuna study, then final training)

In [18]:
!python -m genai.task1 --mode all --trials 25 --trial-epochs 6 --final-epochs 60

2026/10/02 11:49:29 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/10/02 11:49:29 INFO mlflow.store.db.utils: Updating database tables
[t1/] ep 1/6 loss 0.3003 val_obj 0.5949 ssim 0.504 psnr 17.59 (16s)
[t1/] ep 2/6 loss 0.2539 val_obj 0.5245 ssim 0.561 psnr 19.03 (14s)
[t1/] ep 3/6 loss 0.2353 val_obj 0.4951 ssim 0.587 psnr 19.22 (14s)
[t1/] ep 4/6 loss 0.2215 val_obj 0.4794 ssim 0.603 psnr 19.26 (14s)
[t1/] ep 5/6 loss 0.2144 val_obj 0.4568 ssim 0.620 psnr 19.96 (14s)
[t1/] ep 6/6 loss 0.2102 val_obj 0.4512 ssim 0.625 psnr 20.01 (14s)
[t1/] ep 1/6 loss 0.2668 val_obj 0.6665 ssim 0.446 psnr 16.65 (14s)
[t1/] ep 2/6 loss 0.2138 val_obj 0.6001 ssim 0.496 psnr 18.00 (14s)
[t1/] ep 3/6 loss 0.1993 val_obj 0.5668 ssim 0.525 psnr 18.22 (14s)
[t1/] ep 4/6 loss 0.1901 val_obj 0.5371 ssim 0.546 psnr 18.99 (14s)
[t1/] ep 5/6 loss 0.1853 val_obj 0.5286 ssim 0.552 psnr 19.29 (14s)
[t1/] ep 6/6 loss 0.1850 val_obj 0.5231 ssim 0.557 psnr 19.34 (14s)
[t1/] ep 1/6 loss 0.

## 3. Task 2: classifier, then three specialists

In [19]:
!python -m genai.task2 --part classifier --mode all --trials 20 --trial-epochs 5 --final-epochs 40

[cls] ep 1/5 loss 0.1798 val_acc 0.8818 f1 0.8808 (8s)
[cls] ep 2/5 loss 0.0612 val_acc 0.9620 f1 0.9620 (8s)
[cls] ep 3/5 loss 0.0392 val_acc 0.9728 f1 0.9728 (7s)
[cls] ep 4/5 loss 0.0328 val_acc 0.9715 f1 0.9713 (8s)
[cls] ep 5/5 loss 0.0223 val_acc 0.9878 f1 0.9878 (7s)
[cls] ep 1/5 loss 0.2049 val_acc 0.9321 f1 0.9317 (8s)
[cls] ep 2/5 loss 0.0906 val_acc 0.9565 f1 0.9565 (7s)
[cls] ep 3/5 loss 0.0442 val_acc 0.9620 f1 0.9618 (8s)
[cls] ep 4/5 loss 0.0411 val_acc 0.9755 f1 0.9755 (7s)
[cls] ep 5/5 loss 0.0297 val_acc 0.9783 f1 0.9781 (8s)
[cls] ep 1/5 loss 0.3810 val_acc 0.2500 f1 0.1000 (5s)
[cls] ep 2/5 loss 0.1082 val_acc 0.4334 f1 0.3601 (5s)
[cls] ep 3/5 loss 0.0820 val_acc 0.9375 f1 0.9376 (6s)
[cls] ep 4/5 loss 0.0608 val_acc 0.9755 f1 0.9754 (5s)
[cls] ep 5/5 loss 0.0480 val_acc 0.9796 f1 0.9795 (5s)
[cls] ep 1/5 loss 0.4872 val_acc 0.2500 f1 0.1000 (5s)
[cls] ep 2/5 loss 0.1352 val_acc 0.8981 f1 0.8989 (4s)
[cls] ep 3/5 loss 0.0664 val_acc 0.9307 f1 0.9286 (6s)
[cls] ep 4

In [20]:
!python -m genai.task2 --part specialists --mode all --trials 15 --trial-epochs 4 --final-epochs 40

[t2/salt/] ep 1/4 loss 0.1976 val_obj 0.7473 ssim 0.403 psnr 15.16 (16s)
[t2/salt/] ep 2/4 loss 0.1512 val_obj 0.6093 ssim 0.485 psnr 18.18 (15s)
[t2/salt/] ep 3/4 loss 0.1406 val_obj 0.5778 ssim 0.509 psnr 18.85 (15s)
[t2/salt/] ep 4/4 loss 0.1361 val_obj 0.5683 ssim 0.516 psnr 19.11 (15s)
[t2/blur/] ep 1/4 loss 0.1936 val_obj 0.7685 ssim 0.394 psnr 14.40 (15s)
[t2/blur/] ep 2/4 loss 0.1474 val_obj 0.6296 ssim 0.472 psnr 17.90 (15s)
[t2/blur/] ep 3/4 loss 0.1345 val_obj 0.5813 ssim 0.503 psnr 19.13 (15s)
[t2/blur/] ep 4/4 loss 0.1294 val_obj 0.5558 ssim 0.519 psnr 19.84 (15s)
[t2/occlusion/] ep 1/4 loss 0.2133 val_obj 0.8801 ssim 0.340 psnr 11.68 (15s)
[t2/occlusion/] ep 2/4 loss 0.1720 val_obj 0.7761 ssim 0.389 psnr 13.06 (15s)
[t2/occlusion/] ep 3/4 loss 0.1607 val_obj 0.6678 ssim 0.453 psnr 15.86 (15s)
[t2/occlusion/] ep 4/4 loss 0.1563 val_obj 0.6291 ssim 0.473 psnr 17.13 (15s)
[t2/salt/] ep 1/4 loss 0.2300 val_obj 0.8507 ssim 0.354 psnr 12.52 (15s)
[t2/salt/] ep 2/4 loss 0.1647 v

## 4. Task 3: soft mixture-of-experts (needs Task 2 checkpoints)

In [21]:
!python -m genai.task3 --mode all --trials 20 --trial-epochs 4 --warmup-epochs 2 --final-epochs 30

[moe/warmup] ep 1 loss 0.1019 val_obj 0.2128 gate_acc 0.995 w=[0.28, 0.25, 0.24, 0.23] (18s)
[moe/warmup] ep 2 loss 0.1001 val_obj 0.2162 gate_acc 0.996 w=[0.27, 0.25, 0.24, 0.24] (17s)
[moe/finetune] ep 3 loss 0.1072 val_obj 0.2128 gate_acc 0.996 w=[0.29, 0.24, 0.24, 0.22] (41s)
[moe/finetune] ep 4 loss 0.1037 val_obj 0.2119 gate_acc 0.996 w=[0.29, 0.24, 0.23, 0.23] (42s)
[moe/finetune] ep 5 loss 0.1040 val_obj 0.2113 gate_acc 0.996 w=[0.29, 0.24, 0.23, 0.23] (43s)
[moe/finetune] ep 6 loss 0.1037 val_obj 0.2098 gate_acc 0.995 w=[0.3, 0.24, 0.23, 0.23] (42s)
[moe/warmup] ep 1 loss 0.0673 val_obj 0.2233 gate_acc 0.997 w=[0.25, 0.25, 0.25, 0.25] (18s)
[moe/warmup] ep 2 loss 0.0651 val_obj 0.2234 gate_acc 0.997 w=[0.25, 0.25, 0.25, 0.25] (18s)
[moe/finetune] ep 3 loss 0.0724 val_obj 0.2307 gate_acc 0.996 w=[0.25, 0.25, 0.25, 0.25] (42s)
[moe/finetune] ep 4 loss 0.0711 val_obj 0.2286 gate_acc 0.996 w=[0.25, 0.25, 0.25, 0.25] (42s)
[moe/finetune] ep 5 loss 0.0703 val_obj 0.2280 gate_acc 0.9

## 5. Task 4: conditional GAN

In [22]:
!python -m genai.task4 --mode all --trials 15 --trial-epochs 10 --final-epochs 150 --sample-every 10

[gan] ep 1/10 D(real) 0.594 D(fake) 0.578 G_adv 1.347 G_L1 0.287 | val L1 0.1295 SSIM 0.439 (11s)
[gan] ep 2/10 D(real) 0.594 D(fake) 0.555 G_adv 1.263 G_L1 0.260 | val L1 0.1303 SSIM 0.451 (11s)
[gan] ep 3/10 D(real) 0.548 D(fake) 0.537 G_adv 1.457 G_L1 0.256 | val L1 0.1195 SSIM 0.472 (11s)
[gan] ep 4/10 D(real) 0.553 D(fake) 0.532 G_adv 1.536 G_L1 0.253 | val L1 0.1280 SSIM 0.442 (11s)
[gan] ep 5/10 D(real) 0.556 D(fake) 0.537 G_adv 1.483 G_L1 0.251 | val L1 0.1202 SSIM 0.451 (11s)
[gan] ep 6/10 D(real) 0.542 D(fake) 0.516 G_adv 1.575 G_L1 0.247 | val L1 0.1230 SSIM 0.441 (11s)
[gan] ep 7/10 D(real) 0.530 D(fake) 0.512 G_adv 1.670 G_L1 0.243 | val L1 0.1225 SSIM 0.467 (11s)
[gan] ep 8/10 D(real) 0.522 D(fake) 0.486 G_adv 1.734 G_L1 0.241 | val L1 0.1233 SSIM 0.450 (11s)
[gan] ep 9/10 D(real) 0.520 D(fake) 0.503 G_adv 1.772 G_L1 0.239 | val L1 0.1176 SSIM 0.474 (11s)
[gan] ep 10/10 D(real) 0.519 D(fake) 0.497 G_adv 1.748 G_L1 0.236 | val L1 0.1133 SSIM 0.486 (11s)
[gan] ep 1/10 D(rea

## 6. Evaluation on the held-out test sets, ONNX export, app samples

In [23]:
!python -m genai.evaluate --task all
!python -m genai.export
!python scripts/make_samples.py
!python scripts/make_diagrams.py
!python -m genai.report_assets

Task 1: {'clean': 24.94, 'salt': 24.76, 'blur': 24.84, 'occlusion': 21.3}
Task 2: acc 0.9989 oracle 23.33 pred 23.33
Task 3: {'clean': 53.64, 'salt': 25.18, 'blur': 25.62, 'occlusion': 21.84} mixed {'input_psnr': 14.71, 'universal_ae_psnr': 19.42, 'hard_routed_pred_psnr': 18.88, 'soft_moe_psnr': 19.07}
Task 4: {'psnr': 15.608819961547852, 'ssim': 0.5378580689430237, 'l1': 0.10563845187425613} baseline {'psnr': 6.728691101074219, 'ssim': 0.28908970952033997, 'l1': 0.41554680466651917}
/content/repo/src/genai/export.py:53: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(model, args, str(path), dynamo=False, **kw)
[onnx] universal_udae        

## 7. Look at the experiment tracking and package the results

In [24]:
import json; print(json.dumps(json.load(open('outputs/onnx_parity.json')), indent=1)[:1500])

[
 {
  "model": "universal_udae",
  "file": "universal_udae.onnx",
  "max_abs_diff": 5.066394805908203e-07,
  "tolerance": 0.001,
  "ok": true,
  "size_mb": 16.05
 },
 {
  "model": "classifier",
  "file": "classifier.onnx",
  "max_abs_diff": 9.313225746154785e-10,
  "tolerance": 0.001,
  "ok": true,
  "size_mb": 1.18
 },
 {
  "model": "specialist_salt",
  "file": "specialist_salt.onnx",
  "max_abs_diff": 2.980232238769531e-07,
  "tolerance": 0.001,
  "ok": true,
  "size_mb": 16.15
 },
 {
  "model": "specialist_blur",
  "file": "specialist_blur.onnx",
  "max_abs_diff": 4.470348358154297e-07,
  "tolerance": 0.001,
  "ok": true,
  "size_mb": 16.15
 },
 {
  "model": "specialist_occlusion",
  "file": "specialist_occlusion.onnx",
  "max_abs_diff": 3.5762786865234375e-07,
  "tolerance": 0.001,
  "ok": true,
  "size_mb": 16.15
 },
 {
  "model": "soft_moe",
  "file": "soft_moe.onnx",
  "max_abs_diff": 4.172325134277344e-07,
  "tolerance": 0.001,
  "ok": true,
  "size_mb": 49.65
 },
 {
  "model"

In [25]:
# Zip what is needed locally: ONNX models, results, figures, optuna summaries, report assets, samples
!cd /content/repo && zip -qr $WORK/results_bundle.zip models outputs/results outputs/figures outputs/optuna outputs/*.json report/generated report/figures app/backend/samples mlflow.db
print('Download', f'{WORK}/results_bundle.zip', 'from Drive (right-click > Download)')

Download /content/drive/MyDrive/genai_a1/results_bundle.zip from Drive (right-click > Download)
